# Lab 03: Vector Spaces, Projection และ Least Squares
> CLO1 | LLo: อธิบาย Vector Space, Subspace, Rank และคำนวณ Projection / Least Squares ได้

**วิชา**: 1145 201 คณิตศาสตร์สำหรับวิทยาการข้อมูล | **Strang Reference**: Ch.3, Ch.4

---

## บทนำ

สัปดาห์นี้เราจะเรียนรู้แนวคิดที่สำคัญที่สุดใน Linear Algebra สำหรับ Data Science นั่นคือ **Vector Space** และ **Least Squares** ซึ่งเป็นพื้นฐานทางคณิตศาสตร์ของ Linear Regression ทุกประเภท เมื่อ Ax = b ไม่มีคำตอบที่แม่นยำ (เพราะ n > p) เราหาคำตอบที่ 'ดีที่สุด' โดยการ minimize ||Ax - b||² ซึ่งนำไปสู่ **Normal Equations**: AᵀAx̂ = Aᵀb เป้าหมายของ Lab นี้คือให้นักศึกษาเข้าใจว่า Null Space, Column Space, และ Projection เชื่อมกันอย่างไร รวมถึง implement Least Squares ทั้งจาก scratch และด้วย NumPy มี 3 TODOs: หา rank/null space, projection matrix, และ fit เส้นตรงด้วย least squares

In [ ]:
# ─── Import libraries ────────────────────────────────────────────
# วัตถุประสงค์: โหลด library สำหรับ linear algebra และ visualization

import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg

np.set_printoptions(precision=4, suppress=True)
print('NumPy:', np.__version__)

## Part 1: Rank, Column Space, Null Space

**Part นี้เราจะหา rank ของ matrix และ vector ใน Null Space เพื่อเข้าใจว่า Ax = 0 มี solution อะไรบ้าง**

Rank ของ A บอกจำนวน independent columns (หรือ rows) — ถ้า rank < n แสดงว่ามี free variables และ Null Space ไม่ใช่แค่ {0}

In [ ]:
# ─── Demo: Rank และ Null Space ──────────────────────────────────────
# วัตถุประสงค์: แสดงว่า rank-deficient matrix มี non-trivial null space

# Full rank matrix
A_full = np.array([[1, 0, 0],
                   [0, 1, 0],
                   [0, 0, 1]], dtype=float)
print('=== Full Rank ===')
print('rank(A_full) =', np.linalg.matrix_rank(A_full))  # 3
print('Null space dim = n - rank =', 3 - np.linalg.matrix_rank(A_full))  # 0

# Rank-deficient matrix
A_rank2 = np.array([[1, 2, 3],
                    [2, 4, 6],   # row 2 = 2 × row 1
                    [0, 1, 2]], dtype=float)
print('\n=== Rank Deficient ===')
print('A_rank2 =\n', A_rank2)
r = np.linalg.matrix_rank(A_rank2)
print('rank(A_rank2) =', r)  # 2
print('Null space dimension =', 3 - r)  # 1

# หา null space vector ด้วย SVD
_, _, Vt = np.linalg.svd(A_rank2)
null_vec = Vt[-1]  # last row of Vt = null space basis
print('\nNull space vector:', null_vec)
print('Verify A @ null_vec ≈ 0:', np.allclose(A_rank2 @ null_vec, 0))

### 🎯 TODO 1: วิเคราะห์ Feature Matrix ของ Housing Dataset (ระดับ: Easy)

เราต้องการตรวจสอบว่า **feature matrix** ของ housing dataset มี multicollinearity หรือไม่ก่อนนำไปสร้าง Linear Regression model — ถ้า rank < p แสดงว่ามี collinear features

```python
# Housing dataset (n=6, p=4): area, bedrooms, bathrooms, age
X = np.array([[150, 3, 2,  8],
              [200, 4, 2,  5],
              [120, 2, 1, 15],
              [180, 3, 2,  3],
              [160, 3, 2, 10],
              [240, 4, 3,  2]])
```

ให้คุณ:
1. สร้าง X และคำนวณ rank
2. ตรวจสอบว่า bathrooms ≈ bedrooms - 1 หรือไม่ (possible multicollinearity)
3. ลบ bathrooms column ออก → X_reduced (n×3) และหา rank ใหม่
4. คำนวณ Gram matrix **XᵀX** — ถ้า det ≈ 0 แสดงว่าปัญหา multicollinearity

In [ ]:
# TODO 1: เขียน code ที่นี่
# Hint: X.shape, np.linalg.matrix_rank(X)
# Hint: X[:, 2] คือ bathrooms column, X[:, 1] คือ bedrooms
# Hint: X_reduced = np.delete(X, 2, axis=1) ลบ column index 2
# Hint: gram = X.T @ X, det = np.linalg.det(gram)

raise NotImplementedError('กรุณาเติม code ใน TODO 1')

## Part 2: Projection Matrix

**Part นี้เราจะสร้าง Projection Matrix P และใช้ project vector ลงบน subspace เพื่อเข้าใจว่า Least Squares คือการ project b ลงบน Column Space ของ A**

**Projection onto subspace C(A)**: P = A(AᵀA)⁻¹Aᵀ
- Pb = projection of b onto C(A)
- (I - P)b = error term ที่ตั้งฉากกับ C(A)

In [ ]:
# ─── Demo: Projection Matrix ──────────────────────────────────────
# วัตถุประสงค์: สร้าง P = A(AᵀA)⁻¹Aᵀ และ verify properties

# Project onto column space of A
A = np.array([[1, 0],
              [1, 1],
              [1, 2]], dtype=float)  # basis vectors ใน R³

# Projection matrix P = A(AᵀA)⁻¹Aᵀ
P = A @ np.linalg.inv(A.T @ A) @ A.T
print('P (3x3) =\n', P)

# Verify properties:
# 1. P² = P (idempotent)
print('\nP² == P?', np.allclose(P @ P, P))  # True
# 2. Pᵀ = P (symmetric)
print('Pᵀ == P?', np.allclose(P.T, P))  # True
# 3. rank(P) = number of basis vectors
print('rank(P) =', np.linalg.matrix_rank(P))  # 2

# Project a vector b onto C(A)
b = np.array([1, 2, 2], dtype=float)
p_b = P @ b           # projection
e_b = b - p_b         # error (perpendicular to C(A))
print('\nb =', b)
print('Projection p =', p_b)
print('Error e =', e_b)
print('p ⊥ e?', np.allclose(np.dot(p_b, e_b), 0))  # True

### 🎯 TODO 2: Projection ใน 3D (ระดับ: Medium)

เราต้องการ project จุด b = (1, 2, 3) ลงบน plane ที่ span โดยเวกเตอร์ a₁ = (1, 1, 0) และ a₂ = (0, 1, 1) เพื่อเห็นภาพเรขาคณิตของ Least Squares ก่อนที่จะนำไปใช้กับ regression

ให้คุณ:
1. สร้าง matrix A ที่มี a₁, a₂ เป็น columns
2. คำนวณ Projection matrix P = A(AᵀA)⁻¹Aᵀ
3. คำนวณ projection **p** = Pb และ error **e** = b - p
4. Verify: e ⊥ a₁ และ e ⊥ a₂ (ด้วย `np.dot(e, a1) ≈ 0`)
5. Visualize ด้วย matplotlib 3D plot แสดง b, p, e เป็น vectors

In [ ]:
# TODO 2: Projection ใน 3D
# Hint: A = np.column_stack([a1, a2])
# Hint: P = A @ np.linalg.inv(A.T @ A) @ A.T
# Hint: สำหรับ 3D plot: from mpl_toolkits.mplot3d import Axes3D
# Hint: ax.quiver(ox, oy, oz, vx, vy, vz) เพื่อวาด vector

raise NotImplementedError('กรุณาเติม code ใน TODO 2')

## Part 3: Least Squares — Fit a Line

**Part นี้เราจะใช้ Least Squares fit เส้นตรงผ่าน data points เพื่อแสดงว่า Linear Regression คือ Least Squares บน design matrix**

สมการเส้นตรง: y = β₀ + β₁x → **Aβ = y** (overdetermined: n > p)
- Normal Equations: **AᵀA β̂ = Aᵀy**
- Solution: **β̂ = (AᵀA)⁻¹Aᵀy** = `np.linalg.lstsq(A, y)`

In [ ]:
# ─── Demo: Least Squares Line Fit ──────────────────────────────────
# วัตถุประสงค์: fit เส้นตรง y = β₀ + β₁x ด้วย Normal Equations
# แสดงว่า Linear Regression คือ least squares บน design matrix

# Data: x = study hours, y = exam score
x = np.array([2, 3, 5, 7, 8, 10], dtype=float)
y = np.array([45, 55, 65, 75, 80, 90], dtype=float)

# สร้าง design matrix A = [1, x]
# วัตถุประสงค์: column แรก = intercept β₀, column สอง = slope β₁
A = np.column_stack([np.ones_like(x), x])
print('Design matrix A =\n', A)

# วิธีที่ 1: Normal Equations
beta_normal = np.linalg.inv(A.T @ A) @ A.T @ y
print('\nNormal Equations: β =', beta_normal)

# วิธีที่ 2: np.linalg.lstsq (ใช้ SVD — numerically stable กว่า)
beta_lstsq, residuals, rank, sv = np.linalg.lstsq(A, y, rcond=None)
print('lstsq:            β =', beta_lstsq)
print('ผลเท่ากัน?', np.allclose(beta_normal, beta_lstsq))

# Plot
x_line = np.linspace(0, 12, 100)
y_pred = beta_lstsq[0] + beta_lstsq[1] * x_line

plt.figure(figsize=(8, 5))
plt.scatter(x, y, color='red', s=80, zorder=5, label='Data points')
plt.plot(x_line, y_pred, 'b-', label=f'Fit: y = {beta_lstsq[0]:.1f} + {beta_lstsq[1]:.1f}x')
plt.xlabel('Study Hours')
plt.ylabel('Exam Score')
plt.title('Least Squares Line Fit')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🎯 TODO 3: QR Decomposition และ Least Squares (ระดับ: Hard)

เราต้องการ fit **polynomial** y = β₀ + β₁x + β₂x² กับ data ด้วย QR decomposition เพื่อแสดงว่า QR stable กว่า Normal Equations เมื่อ matrix A มี condition number สูง (ill-conditioned)

Data:
```python
x = np.array([0, 1, 2, 3, 4, 5], dtype=float)
y = np.array([1, 3, 7, 13, 21, 31], dtype=float)  # ≈ x² + x + 1
```

ให้คุณ:
1. สร้าง design matrix **A** สำหรับ polynomial degree 2: A = [1, x, x²]
2. QR decomposition: `Q, R = np.linalg.qr(A)`
3. Solve least squares via QR: β̂ = R⁻¹Qᵀy (ใช้ `np.linalg.solve(R, Q.T @ y)`)
4. Compare กับ `np.linalg.lstsq(A, y)`
5. Plot: data points + fitted polynomial curve
6. Print: condition number ของ A และ R (`np.linalg.cond`) — R ควรมี cond number ต่ำกว่า A²

In [ ]:
# TODO 3: QR Decomposition Least Squares
# Hint: design matrix A = np.column_stack([np.ones_like(x), x, x**2])
# Hint: Q, R = np.linalg.qr(A)
# Hint: beta = np.linalg.solve(R, Q.T @ y)
# Hint: np.linalg.cond(A) vs np.linalg.cond(R)
# Hint: fitted curve: y_fit = beta[0] + beta[1]*x_fine + beta[2]*x_fine**2

raise NotImplementedError('กรุณาเติม code ใน TODO 3')

## Reflection Questions

**คำถามที่ 1**: Normal Equations (AᵀAβ̂ = Aᵀy) และ QR method ให้ผลเหมือนกัน แต่ทำไม QR ถึง preferred สำหรับ numerical computation? (Hint: condition number ของ AᵀA vs R)

*(เขียนคำตอบที่นี่)*

---

**คำถามที่ 2**: ถ้า Ax = b ไม่มี exact solution (overdetermined system) Least Squares หา x̂ ที่ minimize ||Ax̂ - b||² แต่ถ้ามีหลาย x̂ ที่ให้ค่า minimum เท่ากัน (underdetermined) NumPy lstsq จะเลือก x̂ ใด?

*(เขียนคำตอบที่นี่)*

---

**คำถามที่ 3**: ใน TODO 1 ถ้าเราไม่ลบ bathrooms column ออก จะเกิดอะไรขึ้นกับ det(XᵀX)? และนักวิทยาการข้อมูลควรจัดการ multicollinearity อย่างไรในทางปฏิบัติ?

*(เขียนคำตอบที่นี่)*